In [2]:
import pandas as pd
import numpy as np
import pickle
import joblib

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score

# Tabular

### Loading Tabular Dataset

In [ ]:
mineral_cols = ["AU", "AG", "CU", "CO", "NI"]
target_cols = ["target_AU", "target_AG", "target_CU", "target_CO", "target_NI"]
target_col_string = "target_AU"
target_col = [target_col_string] # For single target prediction

df_processed = pd.read_csv('../processed(500)_geology_ml_training_data.csv', low_memory=False)
df_processed = df_processed[df_processed["hasImage"]] # Only get rows that have images
df_processed = df_processed.dropna(subset=target_col).copy()
df_processed[target_col_string] = (df_processed[target_col_string] > 0).astype(np.float32)

tabular_cols = [col for col in df_processed.columns if col not in  mineral_cols + target_cols + ['spatial_cluster', 'CODE_LITH', 'STRAT', 'Sample_ID', 'hasImage']]

bad_cols = df_processed[tabular_cols].select_dtypes(include=["object"]).columns
print(bad_cols)

df_processed[tabular_cols].head()

Index([], dtype='object')


,Easting,Northing,dist_fault,dist_cont,dist_fault_norm,dist_cont_norm,AU_missing,AG_missing,CU_missing,CO_missing,...,V3,V3A,V3B,V3D,V3F,V3G,V3H,V4,V4A,V4FO
4,-823281.410385,445223.349511,8620.008852,708.354658,0.040018,0.004530,0.000000,0.000000,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
6,-822577.690915,449159.895115,5937.687683,8.809315,0.027565,0.000056,0.333333,0.333333,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
7,-822498.982021,419697.736482,7963.856827,225.416115,0.036971,0.001442,0.000000,1.000000,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
8,-822352.108967,421386.167458,8951.045298,317.167447,0.041554,0.002028,0.000000,1.000000,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0
9,-822343.618110,420346.833544,8246.925134,289.456545,0.038286,0.001851,0.000000,0.000000,0.0,0.0,...,0,0,0,0,0,0,0,0,0,0


### Splitting Tabular Dataset

In [ ]:
# Splitting dataframe
train_df, test_df = train_test_split(df_processed, test_size=0.15, stratify=df_processed["target_AU"], random_state=42)

X = train_df[tabular_cols]
y = train_df[target_col_string].astype(int)

# Stratified K-Folds
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf.split(X, y))

## Training Model

In [21]:
fold_results = []

for fold, (train_idx, val_idx) in enumerate(folds):

    X_train = X.iloc[train_idx]
    X_val   = X.iloc[val_idx]

    y_train = y.iloc[train_idx]
    y_val   = y.iloc[val_idx]

    model = Pipeline([
        ("scaler", StandardScaler()),
        ("logreg", LogisticRegression(
            max_iter=2000,
            class_weight="balanced",
            solver="lbfgs",
            random_state=42
        ))
    ])

    model.fit(X_train, y_train)
    joblib.dump(model, f"logreg_fold{fold}.joblib")

    # probability of positive class
    y_prob = model.predict_proba(X_val)[:, 1]

    roc = roc_auc_score(y_val, y_prob)
    pr  = average_precision_score(y_val, y_prob)

    fold_results.append({
        "fold": fold,
        "roc_auc": roc,
        "pr_auc": pr,
        "positive_rate": y_val.mean()
    })

    print(f"Fold {fold}")
    print(f"ROC AUC: {roc:.4f}")
    print(f"PR AUC:  {pr:.4f}")
    print(f"Positive rate: {y_val.mean():.4f}")
    print("-" * 30)

results_df = pd.DataFrame(fold_results)
results_df.to_csv("logistic_regression_5fold_results.csv", index=False)
print(results_df)
print("Mean ROC AUC:", results_df["roc_auc"].mean())
print("Mean PR AUC:", results_df["pr_auc"].mean())


Fold 0
ROC AUC: 0.7611
PR AUC:  0.1506
Positive rate: 0.0320
------------------------------
Fold 1
ROC AUC: 0.7835
PR AUC:  0.1815
Positive rate: 0.0316
------------------------------
Fold 2
ROC AUC: 0.7964
PR AUC:  0.2627
Positive rate: 0.0316
------------------------------
Fold 3
ROC AUC: 0.7744
PR AUC:  0.1688
Positive rate: 0.0316
------------------------------
Fold 4
ROC AUC: 0.7812
PR AUC:  0.1599
Positive rate: 0.0316
------------------------------
   fold   roc_auc    pr_auc  positive_rate
0     0  0.761096  0.150626       0.031970
1     1  0.783450  0.181481       0.031637
2     2  0.796350  0.262660       0.031637
3     3  0.774445  0.168845       0.031637
4     4  0.781235  0.159934       0.031637
Mean ROC AUC: 0.7793152117466742
Mean PR AUC: 0.18470903937017022


In [3]:
results_df = pd.read_csv("logistic_regression_5fold_results.csv")

print(results_df["roc_auc"].mean())
print(results_df["roc_auc"].std())

print(results_df["pr_auc"].mean())
print(results_df["pr_auc"].std())

0.7793152117466742
0.01290943426732727
0.1847090393701702
0.04503851334509959
